# Evaluate & Compare Models

Reads what `01_train_models.ipynb` produced (`results/runs.jsonl` + `checkpoints/<model_name>/`) — **this notebook never retrains anything**, so re-running it to try a different evaluation subset size or decoding strategy is fast and doesn't touch a GPU for training.

For each trained model, this: picks its best (lowest-val-loss) checkpoint, scores it on a test-set sample with Valid Syntax Rate and Best-IoU under three decoding strategies (greedy / beam / syntax-repair), builds a comparison table + chart, and shows qualitative predictions on a few examples.

> **Methodology note**: the original notebook's often-quoted "~50% VSR, ~0.78 mean IoU" was computed on **2 test samples** (a quick sanity check, not a benchmark). `EVAL_SAMPLE_SIZE` below defaults to a much larger random subset — still not the full multi-thousand-example test set (beam search is slow per sample), but large enough that the numbers mean something.

In [ ]:
import os, pathlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/adebowalep/image-to-cadquery.git"  # <-- set this (private repo: https://<token>@github.com/...)
    repo_dir = pathlib.Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

    # Colab already ships torch/torchvision/transformers/pandas/matplotlib -- install only what it lacks.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cadquery==2.6.0", "cadquery-ocp==7.8.1.1", "trimesh", "datasets"],
        check=True,
    )

_REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_REPO_ROOT))          # `metrics/`
sys.path.insert(0, str(_REPO_ROOT / "src"))  # `cad_code_gen` (no `pip install -e` needed)

# Everything that must outlive the session (dataset cache, tokenizer, checkpoints, results)
# lives under STORAGE_ROOT: Google Drive on Colab, the repo itself locally.
STORAGE_ROOT = pathlib.Path("/content/drive/MyDrive/cad_code_gen") if IN_COLAB else _REPO_ROOT
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("repo:", _REPO_ROOT, "| storage:", STORAGE_ROOT)

In [ ]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

from cad_code_gen.data import CADCodeDataset, build_image_transform, collate_batch
from cad_code_gen.decoding import decode_beam, decode_greedy, decode_with_syntax_repair, fix_floats
from cad_code_gen.experiment_log import best_epoch_by_val_loss, load_runs
from cad_code_gen.registry import MODEL_REGISTRY
from cad_code_gen.tokenizer import load_tokenizer
from metrics.best_iou import get_iou_best
from metrics.valid_syntax_rate import evaluate_syntax_rate_simple

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Config

In [ ]:
SEED = 42
EVAL_SAMPLE_SIZE = 200   # random test-set subset; raise if you have the GPU time
MAX_DECODE_LEN = 256
BEAM_SIZE = 3
N_QUALITATIVE_EXAMPLES = 3

TOKENIZER_DIR = STORAGE_ROOT / "cadquery_tokenizer"
RESULTS_LOG = STORAGE_ROOT / "results" / "runs.jsonl"

## Load test set & tokenizer, pick each model's best checkpoint

In [ ]:
ds_test_full = load_dataset("CADCODER/GenCAD-Code", split="test", cache_dir=str(STORAGE_ROOT / "huggingface_cache"))
ds_test = ds_test_full.shuffle(seed=SEED).select(range(min(EVAL_SAMPLE_SIZE, len(ds_test_full))))
tokenizer = load_tokenizer(TOKENIZER_DIR)
print(f"Evaluating on {len(ds_test)} of {len(ds_test_full)} test examples")

In [ ]:
runs_df = load_runs(RESULTS_LOG)
best_checkpoints = {
    model_name: best_epoch_by_val_loss(group.to_dict("records"))
    for model_name, group in runs_df.groupby("model")
}
pd.DataFrame(best_checkpoints).T

## Evaluate every model x decoding strategy

`decode_with_syntax_repair` beam-searches a few candidates and keeps whichever one is valid Python (see `src/cad_code_gen/decoding.py`) — comparing it against plain greedy/beam decoding shows how much of the VSR gap that alone closes, with no retraining.

In [ ]:
img_transform = build_image_transform()
test_ds = CADCodeDataset(ds_test, tokenizer, max_len=MAX_DECODE_LEN, img_transform=img_transform)
ground_truth = {str(row["deepcad_id"]): row["cadquery"] for row in ds_test}

def score(pred_codes_by_id: dict) -> dict:
    vsr = evaluate_syntax_rate_simple(pred_codes_by_id)
    ious = []
    for sample_id, pred_code in pred_codes_by_id.items():
        try:
            ious.append(get_iou_best(ground_truth[sample_id], pred_code))
        except Exception:
            pass  # invalid/non-executable code: contributes 0 to VSR, not to mean IoU
    return {"vsr": vsr, "mean_iou_best": sum(ious) / len(ious) if ious else 0.0}

In [ ]:
comparison_rows = []

for model_name, best in best_checkpoints.items():
    print(f"\n{'='*20} {model_name} (epoch {best['epoch']}) {'='*20}")
    build_kwargs = {"pretrained_backbone": False} if model_name in ("baseline", "spatial") else {}
    model = MODEL_REGISTRY[model_name].build(
        tokenizer.vocab_size, tokenizer.pad_token_id, **build_kwargs
    ).to(DEVICE)
    model.load_state_dict(torch.load(best["checkpoint"], map_location=DEVICE))
    model.eval()

    greedy_preds, repair_preds = {}, {}
    for i in range(len(test_ds)):
        sample = test_ds[i]
        image = sample["image"].unsqueeze(0).to(DEVICE)
        sample_id = str(sample["id"])

        greedy_preds[sample_id] = fix_floats(decode_greedy(model, tokenizer, image, max_len=MAX_DECODE_LEN)[0])
        repair_preds[sample_id] = decode_with_syntax_repair(
            model, tokenizer, image.squeeze(0), max_len=MAX_DECODE_LEN, beam_size=BEAM_SIZE
        )

    for strategy_name, preds in [("greedy", greedy_preds), ("syntax_repair", repair_preds)]:
        metrics = score(preds)
        print(f"  {strategy_name:14s} VSR={metrics['vsr']:.3f}  mean IoU={metrics['mean_iou_best']:.3f}")
        comparison_rows.append({"model": model_name, "decoding": strategy_name, **metrics})

## Comparison table & chart

In [ ]:
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric, title in [(axes[0], "vsr", "Valid Syntax Rate"), (axes[1], "mean_iou_best", "Mean Best-IoU")]:
    pivot = comparison_df.pivot(index="model", columns="decoding", values=metric)
    pivot.plot.bar(ax=ax)
    ax.set_title(title); ax.set_ylabel(metric); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Qualitative inference

A handful of test images with each model's syntax-repaired prediction next to the ground truth, for a quick sanity check beyond the aggregate numbers above.

In [ ]:
sample_indices = range(min(N_QUALITATIVE_EXAMPLES, len(test_ds)))

for idx in sample_indices:
    sample = test_ds[idx]
    gt_code = sample["code_string"]

    plt.figure(figsize=(3, 3))
    plt.imshow(sample["image"].permute(1, 2, 0).numpy() * 0.229 + 0.485)  # rough unnormalize for viewing
    plt.axis("off"); plt.title(f"test sample {sample['id']}")
    plt.show()

    print("--- ground truth ---")
    print(gt_code)

    for model_name, best in best_checkpoints.items():
        build_kwargs = {"pretrained_backbone": False} if model_name in ("baseline", "spatial") else {}
        model = MODEL_REGISTRY[model_name].build(
            tokenizer.vocab_size, tokenizer.pad_token_id, **build_kwargs
        ).to(DEVICE)
        model.load_state_dict(torch.load(best["checkpoint"], map_location=DEVICE))
        model.eval()

        image = sample["image"].unsqueeze(0).to(DEVICE)
        pred = decode_with_syntax_repair(model, tokenizer, image.squeeze(0), max_len=MAX_DECODE_LEN, beam_size=BEAM_SIZE)
        print(f"--- {model_name} prediction ---")
        print(pred)
    print("=" * 60)

## Conclusion

Fill this in once the cells above have actually run on real GPU training:
- Which architecture wins on VSR? On IoU? Are they the same model?
- How much does `decode_with_syntax_repair` alone close the VSR gap, independent of architecture?
- Is the winner worth taking into `scripts/finetune_rl.py` for a further RL fine-tuning pass?